In [26]:
import sys
import torch
import transformers

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

Python: /home/saurin/sistema-servicios-ia/backend/venv/bin/python
PyTorch: 2.15.0+cpu
Transformers: 5.22.0


In [27]:
from transformers import AutoTokenizer, AutoModel

NOMBRE_MODELO = "dccuchile/bert-base-spanish-wwm-cased"

tokenizador = AutoTokenizer.from_pretrained(NOMBRE_MODELO)
modelo_beto = AutoModel.from_pretrained(NOMBRE_MODELO)
modelo_beto.eval()

print("BETO cargado")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12428.60it/s]
[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

BETO cargado


In [28]:
texto = "La laptop se calienta mucho y se apaga."

entradas = tokenizador(
    texto,
    return_tensors="pt",
    truncation=True,
    max_length=128,
)

with torch.inference_mode():
    salida = modelo_beto(**entradas)

print("Tokens:", tokenizador.convert_ids_to_tokens(entradas["input_ids"][0].tolist()))
print("Forma de la salida:", salida.last_hidden_state.shape)

Tokens: ['[CLS]', 'La', 'la', '##pto', '##p', 'se', 'cali', '##enta', 'mucho', 'y', 'se', 'apag', '##a', '.', '[SEP]']
Forma de la salida: torch.Size([1, 15, 768])


In [29]:
vectores_tokens = salida.last_hidden_state

mascara = entradas["attention_mask"].unsqueeze(-1)
mascara = mascara.to(vectores_tokens.dtype)

vector_frase = (
    (vectores_tokens * mascara).sum(dim=1)
    / mascara.sum(dim=1).clamp(min=1)
)

print("Forma del vector:", vector_frase.shape)
print("Primeros 5 valores:", vector_frase[0, :5])

Forma del vector: torch.Size([1, 768])
Primeros 5 valores: tensor([-0.2003,  0.1856, -0.0799,  0.1297, -0.0602])


In [30]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

RAIZ = Path("/home/saurin/sistema-servicios-ia")
df = pd.read_csv(RAIZ / "ml/data/ordenes.csv")

X_train, X_test, y_train, y_test = train_test_split(
    df["descripcion_falla"],
    df["categoria_falla"],
    test_size=0.2,
    random_state=42,
    stratify=df["categoria_falla"],
)

print("Entrenamiento:", len(X_train))
print("Prueba:", len(X_test))

Entrenamiento: 3200
Prueba: 800


In [31]:
import numpy as np
from time import perf_counter

def obtener_vectores(textos, lote=16):
    textos = list(textos)
    bloques = []
    inicio = perf_counter()

    modelo_beto.eval()

    with torch.inference_mode():
        for desde in range(0, len(textos), lote):
            grupo = textos[desde:desde + lote]

            entradas_lote = tokenizador(
                grupo,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            )

            salida_lote = modelo_beto(**entradas_lote)
            tokens = salida_lote.last_hidden_state

            mascara = entradas_lote["attention_mask"].unsqueeze(-1)
            mascara = mascara.to(tokens.dtype)

            promedios = (
                (tokens * mascara).sum(dim=1)
                / mascara.sum(dim=1).clamp(min=1)
            )

            bloques.append(promedios.cpu().numpy())

            procesadas = min(desde + lote, len(textos))
            if procesadas % (lote * 10) == 0 or procesadas == len(textos):
                segundos = perf_counter() - inicio
                print(
                    f"{procesadas}/{len(textos)} frases "
                    f"— {segundos:.1f} segundos",
                    flush=True,
                )

    return np.concatenate(bloques, axis=0)

In [32]:
print("Procesando entrenamiento...")
vectores_train = obtener_vectores(X_train)

print("\nProcesando prueba...")
vectores_test = obtener_vectores(X_test)

print("\nForma entrenamiento:", vectores_train.shape)
print("Forma prueba:", vectores_test.shape)

Procesando entrenamiento...
160/3200 frases — 2.9 segundos
320/3200 frases — 5.9 segundos
480/3200 frases — 9.7 segundos
640/3200 frases — 13.8 segundos
800/3200 frases — 17.1 segundos
960/3200 frases — 20.6 segundos
1120/3200 frases — 24.6 segundos
1280/3200 frases — 27.7 segundos
1440/3200 frases — 31.0 segundos
1600/3200 frases — 34.8 segundos
1760/3200 frases — 38.6 segundos
1920/3200 frases — 42.0 segundos
2080/3200 frases — 45.6 segundos
2240/3200 frases — 49.2 segundos
2400/3200 frases — 52.6 segundos
2560/3200 frases — 55.9 segundos
2720/3200 frases — 59.3 segundos
2880/3200 frases — 62.5 segundos
3040/3200 frases — 66.0 segundos
3200/3200 frases — 69.9 segundos

Procesando prueba...
160/800 frases — 3.8 segundos
320/800 frases — 7.5 segundos
480/800 frases — 10.8 segundos
640/800 frases — 13.8 segundos
800/800 frases — 17.4 segundos

Forma entrenamiento: (3200, 768)
Forma prueba: (800, 768)


In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clasificador_beto = LogisticRegression(max_iter=2000)

clasificador_beto.fit(vectores_train, y_train)

print("Clasificador entrenado")

Clasificador entrenado


In [40]:
pred_beto = clasificador_beto.predict(vectores_test)

print(
    f"Exactitud en prueba sintética (BETO): "
    f"{accuracy_score(y_test, pred_beto):.2%}"
)

print(classification_report(y_test, pred_beto, zero_division=0))

Exactitud en prueba sintética (BETO): 95.88%
                    precision    recall  f1-score   support

      atasco_papel       0.97      0.97      0.97        61
           bateria       0.94      0.98      0.96        64
     cabezal_tinta       1.00      0.97      0.98        66
        disco_duro       1.00      0.93      0.96        84
      fuente_poder       1.00      0.92      0.96        39
       fusor_laser       0.97      0.97      0.97        37
          pantalla       0.90      0.97      0.93        99
       placa_madre       1.00      0.93      0.97        46
sobrecalentamiento       0.97      0.92      0.95        76
          software       0.95      0.98      0.96       176
  teclado_touchpad       0.94      0.94      0.94        52

          accuracy                           0.96       800
         macro avg       0.97      0.95      0.96       800
      weighted avg       0.96      0.96      0.96       800



In [41]:
prueba_manual = pd.read_csv(
    RAIZ / "ml/data/prueba_manual.csv",
    skipinitialspace=True,
)

prueba_manual["categoria"] = prueba_manual["categoria"].str.strip()

vectores_manual = obtener_vectores(prueba_manual["texto"])
pred_manual_beto = clasificador_beto.predict(vectores_manual)

correctas = pred_manual_beto == prueba_manual["categoria"].to_numpy()
aciertos = int(correctas.sum())
total = len(prueba_manual)

print(
    f"\nExactitud en prueba manual (BETO): "
    f"{aciertos / total:.2%} ({aciertos} de {total})"
)

fallos_beto = prueba_manual.assign(predicho=pred_manual_beto)
fallos_beto = fallos_beto.loc[
    fallos_beto["categoria"] != fallos_beto["predicho"],
    ["texto", "categoria", "predicho"],
]

with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(fallos_beto)

29/29 frases — 0.5 segundos

Exactitud en prueba manual (BETO): 58.62% (17 de 29)


,texto,categoria,predicho
0,mi laptop quedó muerta después del apagón de ayer,placa_madre,pantalla
7,se borraron mis documentos y no puedo abrir nada,disco_duro,software
8,la torre no da señales de vida ni luces ni ruido,fuente_poder,placa_madre
9,al encenderla escuché un estallido y se fue la luz del cuarto,fuente_poder,placa_madre
16,el cursor salta de un lado a otro sin que lo toque,teclado_touchpad,fusor_laser
17,derramé jugo sobre el teclado,teclado_touchpad,fusor_laser
18,la compu va lentísima y me aparecen anuncios por todos lados,software,pantalla
19,necesito reinstalar el sistema y los programas,software,disco_duro
20,me pide una clave que no conozco al iniciar windows,software,disco_duro
22,dice paper jam pero la bandeja está vacía,atasco_papel,bateria


In [42]:
print(
    "Exactitud sintética (BETO):",
    f"{accuracy_score(y_test, pred_beto):.2%}",
)

resumen = (
    df.groupby("categoria_falla")
    .agg(
        ejemplos=("descripcion_falla", "size"),
        textos_distintos=("descripcion_falla", "nunique"),
    )
)

resumen["repeticiones_exactas"] = (
    resumen["ejemplos"] - resumen["textos_distintos"]
)

display(resumen)

Exactitud sintética (BETO): 95.88%


,ejemplos,textos_distintos,repeticiones_exactas
categoria_falla,,,
atasco_papel,302,161,141
bateria,320,171,149
cabezal_tinta,332,178,154
disco_duro,419,213,206
fuente_poder,194,107,87
fusor_laser,187,117,70
pantalla,494,245,249
placa_madre,231,129,102
sobrecalentamiento,380,190,190


In [43]:
# Normalizamos únicamente para comparar textos:
# ignoramos mayúsculas y diferencias de espacios.
def normalizar_textos(serie):
    return (
        serie.str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

textos_train = set(normalizar_textos(X_train))
textos_test = normalizar_textos(X_test)

vistas = textos_test.isin(textos_train).to_numpy()

print(
    f"Frases de prueba presentes en entrenamiento: "
    f"{vistas.sum()} de {len(vistas)} ({vistas.mean():.1%})"
)

etiquetas_test = y_test.to_numpy()

for nombre, mascara in [
    ("Textos ya vistos", vistas),
    ("Textos no vistos exactamente", ~vistas),
]:
    cantidad = int(mascara.sum())

    if cantidad:
        exactitud = accuracy_score(
            etiquetas_test[mascara],
            pred_beto[mascara],
        )
        print(f"{nombre}: {cantidad} ejemplos — {exactitud:.2%}")

Frases de prueba presentes en entrenamiento: 504 de 800 (63.0%)
Textos ya vistos: 504 ejemplos — 96.43%
Textos no vistos exactamente: 296 ejemplos — 94.93%


In [45]:
necesarios = ["modelo_beto", "tokenizador", "clasificador_beto"]
faltantes = [nombre for nombre in necesarios if nombre not in globals()]

if faltantes:
    raise RuntimeError(
        f"Faltan objetos en memoria: {faltantes}. "
        "Ejecuta primero las celdas del notebook 03."
    )

In [48]:
from pathlib import Path
from importlib.metadata import version
import json
import joblib

# Acceso directo a los objetos del notebook 03.
# Si alguno falta, el código se detendrá antes de crear archivos.
print("Extractor:", type(modelo_beto).__name__)
print("Tokenizador:", type(tokenizador).__name__)
print("Clasificador:", type(clasificador_beto).__name__)
print("Categorías:", len(clasificador_beto.classes_))

carpeta = Path(
    "/home/saurin/sistema-servicios-ia/ml/models/clasificador_beto_v1"
)

configuracion = {
    "modelo_base": "dccuchile/bert-base-spanish-wwm-cased",
    "max_length": 128,
    "pooling": "promedio_tokens_sin_padding",
    "incluye_tokens_especiales": True,
    "categorias": clasificador_beto.classes_.tolist(),
    "uso": "prototipo_con_datos_sinteticos",
    "versiones": {},
}

for paquete in [
    "torch", "transformers", "scikit-learn", "numpy", "joblib"
]:
    configuracion["versiones"][paquete] = version(paquete)

carpeta.mkdir(parents=True, exist_ok=True)

print("Guardando BETO...")
modelo_beto.save_pretrained(carpeta / "beto")
tokenizador.save_pretrained(carpeta / "beto")

print("Guardando clasificador...")
joblib.dump(
    clasificador_beto,
    carpeta / "clasificador.joblib",
)

print("Guardando configuración...")
with (carpeta / "configuracion.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(configuracion, archivo, ensure_ascii=False, indent=2)

print("\nModelo guardado en:", carpeta)
print("Configuración:", (carpeta / "configuracion.json").is_file())
print("Clasificador:", (carpeta / "clasificador.joblib").is_file())
print("Carpeta BETO:", (carpeta / "beto").is_dir())

Extractor: BertModel
Tokenizador: BertTokenizer
Clasificador: LogisticRegression
Categorías: 11
Guardando BETO...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]

Guardando clasificador...
Guardando configuración...

Modelo guardado en: /home/saurin/sistema-servicios-ia/ml/models/clasificador_beto_v1
Configuración: True
Clasificador: True
Carpeta BETO: True


In [49]:
from pathlib import Path

carpeta = Path(
    "/home/saurin/sistema-servicios-ia/ml/models/clasificador_beto_v1"
)

print("Configuración:", (carpeta / "configuracion.json").is_file())
print("Clasificador:", (carpeta / "clasificador.joblib").is_file())
print("Carpeta BETO:", (carpeta / "beto").is_dir())

Configuración: True
Clasificador: True
Carpeta BETO: True
